<a href="https://colab.research.google.com/github/dzianismr/GetPKOpenFDA/blob/main/OpenFDA_getPK_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is Python script for a simple AI agent which **extracts and tabulates PK information from the FDA labels**. The key steps are:
- search and retrieval of relevant labels via OpenFDA API;
- LLM-based extraction of PK information;

The script uses OpenAI gpt-4o LLM, therefore OPENAI_API_KEY needs to be provided, alternatively a different LLM can be used.

In [ ]:
#set up env

##  instal
!!pip install litellm #
from litellm import completion # Python interface to LLMs

# import
import requests # supports API usage in Python
import json # JSON Parser for Python

# standard data wrangling packages
import pandas as pd
from typing import List, Dict
import time


In [ ]:
# set llm access
# for this to work "secrets" has to be set up in google colab
import os
from google.colab import userdata

# Set up LLM access
api_key = userdata.get('OpenAIAPI')
os.environ['OPENAI_API_KEY'] = api_key

OPENAI_API_KEY = userdata.get('OpenAIAPI')

In [ ]:
# Retrieve labelling information via OpenFDA API
# n labels containing pediatric PK information for monoclonal antibodies will be retrieved

# file name
DRUG = "paediatric_mab"
n = 3

url = "https://api.fda.gov/drug/label.json?search=openfda.generic_name:*mab+AND+pharmacokinetics:pediat*&limit="+ str(n)


# Make the GET request
response = requests.get(url)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    # Parse the JSON response
    data = response.json()
    print("API request successful!")
    # Display the first 3 results for demonstration
    if 'results' in data and len(data['results']) > 0:
        print("First 3 results:")
        for i, result in enumerate(data['results'][:3]):
            print(f"\nResult {i+1}:")
            print(json.dumps(result, indent=2))
    else:
        print("No results found or 'results' key missing.")
else:
    print(f"API request failed with status code: {response.status_code}")
    #print(f"Response: {response.text}") # uncomment to print json output

In [ ]:
# define helper function to interact with LLM
def generate_response(messages: List[Dict]) -> str:
    """Call LLM to get response"""
    response = completion(
        model="openai/gpt-4o",
        messages=messages,
        max_tokens=4096
    )
    return response.choices[0].message.content

# prompts
message1 ={"role": "system", "content": "You an experienced pharmacokineticist with attention to the detail. You specialize in extracting PK parameters from the text. You always return results in JSON format."}
message2 ={"role": "user", "content": "The following text was extracted from FDA labelling document. Extract reported PK parameters e.g. CL, AUC, Cmin (Ctrough), Cmax also extract uncertainty of the estimate as well as units. Extracted parameters should be reported in tabular view. Table columns should include: drug, population, parameter, value, uncertainty_measure, uncertainty_type, unit."}


In [ ]:

all_pk_dfs = []

# iterate through the list of results, compound by compound
if 'results' in data and len(data['results']) > 0:
    print(f"Processing {len(data['results'])} documents...")

    for idx, result in enumerate(data['results']):

        # 1. get Drug Name (Brand or Generic)
        brand_name   = result.get('openfda', {}).get('brand_name', ['Unknown'])[0]
        generic_name = result.get('openfda', {}).get('generic_name', ['Unknown'])[0]
        current_drug_label = generic_name

        print(f"--- Processing Result {idx+1}: {current_drug_label} ---")

        # 2. Extract Pharmacokinetics section
        pk_text_list = result.get("pharmacokinetics") or result.get("12.3 Pharmacokinetics")

        if not pk_text_list:
            print(f"No PK section found for {current_drug_label}. Skipping.")
            continue

        pk_text = " ".join(pk_text_list)

        # 3. Prepare and call LLM
        # create prompt by combining user instruction with the pk section
        updated_message2 = message2.copy()
        updated_message2['content'] = updated_message2.get('content', '') + pk_text

        messages = [
            message1,
            updated_message2
        ]


        try:
            # submit prompt and get response from the LLM
            raw_response = generate_response(messages)
            # Clean JSON response
            clean_json = raw_response.strip().replace('```json', '').replace('```', '')
            parsed_data = json.loads(clean_json)

            # 4. Create DataFrame
            temp_df = pd.DataFrame(parsed_data)
            if not temp_df.empty:
                temp_df['source_brand_name'] = brand_name
                temp_df['source_generic_name'] = generic_name
                all_pk_dfs.append(temp_df)
                print(f"Successfully extracted {len(temp_df)} parameters.")


        except Exception as e:
            print(f"Error processing {current_drug_label}: {e}")



In [ ]:
# 5. Combine all results into a single Long Format table

# Ensure the output directory exists
output_dir = 'out_getPK'
os.makedirs(output_dir, exist_ok=True)

if all_pk_dfs:
    final_long_df = pd.concat(all_pk_dfs, ignore_index=True)
    display(final_long_df)

    # Export the consolidated table
    final_export_path = os.path.join(output_dir, f"all_{DRUG}_labels_PK_long.csv")
    final_long_df.to_csv(final_export_path, index=False)
    print(f"\nFinal consolidated table saved to: {final_export_path}")
else:
    print("No PK data was extracted from any of the documents.")